khushi khatri beb222 exp2


Aim:Apply various text preprocessing techniques fro any given text:tokenization and filteration and script validation

Theory

Raw text data collected from real-world sources (such as user reviews, social media posts, or documents) is unstructured and contains a lot of noise — filler words, grammatical variations, and inconsistent word forms — that add little value to text analysis or Natural Language Processing (NLP) tasks. Text preprocessing is the essential first step in any NLP pipeline that transforms raw text into a clean, standardized format suitable for machine learning models, information retrieval, or text mining. This experiment focuses on two key preprocessing techniques: Stop Word Removal and Stemming/Lemmatization.

1. Stop Word Removal

Stop words are common words in a language (such as "the," "is," "at," "which," "on," "and," "a") that occur very frequently in text but carry little to no meaningful semantic information for tasks like text classification, sentiment analysis, or search indexing. Since these words appear in nearly every sentence regardless of context, retaining them increases the dimensionality of the data without adding discriminative value, and can slow down downstream algorithms.

Stop word removal is the process of filtering out these high-frequency, low-information words from a tokenized text, leaving behind only the content-bearing words. For example:

Original: "The place was very cozy and the check-in was smooth"
After stop word removal: "place cozy check-in smooth"

Libraries like NLTK provide predefined stop word lists for various languages (e.g., nltk.corpus.stopwords). While effective, stop word removal must be applied carefully, since in some contexts (e.g., sentiment analysis, where words like "not" are critical) removing certain stop words can distort meaning.

2. Stemming

Stemming is a rule-based, crude heuristic process that reduces a word to its root or base form (stem) by chopping off prefixes or suffixes, without necessarily producing a valid dictionary word. It is fast and computationally inexpensive, making it suitable for large-scale text processing where speed matters more than linguistic precision.

The most widely used stemming algorithm is the Porter Stemmer, which applies a series of suffix-stripping rules. For example:

"amazing" → "amaz"
"studies" → "studi"
"connection," "connected," "connecting" → "connect"

Because stemming operates purely on word surface patterns, it can sometimes produce non-words or over-stem/under-stem (e.g., merging unrelated words or failing to reduce related words to the same stem), reducing linguistic accuracy.

3. Lemmatization

Lemmatization is a more linguistically informed process that reduces a word to its lemma — its proper dictionary/base form — by using vocabulary and morphological analysis (often with the help of Part-of-Speech, or POS, tagging) rather than simple suffix stripping. Unlike stemming, lemmatization always produces valid words.

For example:

"amazing" (adjective) → "amazing"
"studies" (verb) → "study"
"better" (adjective, with POS context) → "good"

Lemmatization typically uses lexical databases such as WordNet, and its accuracy improves significantly when combined with POS tagging, since the correct lemma of a word often depends on its grammatical role in the sentence (e.g., "saw" as a noun vs. a verb).

In [2]:
# !pip install nltk pandas

import re
import string
import pandas as pd
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print('Libraries loaded successfully.')

Libraries loaded successfully.


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [4]:
def tokenize_text(text):
    """Return sentence tokens and word tokens for a given piece of text."""
    if not isinstance(text, str) or text.strip() == '':
        return [], []
    sentences = sent_tokenize(text)
    words = word_tokenize(text)
    return sentences, words

# Demo on a single sample review
sample_text = df['review_text'].iloc[0]
sent_tokens, word_tokens = tokenize_text(sample_text)

print('Original Text:\n', sample_text)
print('\nSentence Tokens:\n', sent_tokens)
print('\nWord Tokens:\n', word_tokens)

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed.

Sentence Tokens:
 ['Amazing stay!', 'The place felt very cozy for 4 guests.', 'Check-in was smooth and the amenities were exactly what we needed.']

Word Tokens:
 ['Amazing', 'stay', '!', 'The', 'place', 'felt', 'very', 'cozy', 'for', '4', 'guests', '.', 'Check-in', 'was', 'smooth', 'and', 'the', 'amenities', 'were', 'exactly', 'what', 'we', 'needed', '.']


In [5]:
def filter_tokens(tokens):
    """Filter word tokens: lowercase, remove punctuation, numbers, stopwords, short tokens."""
    cleaned = []
    for tok in tokens:
        tok = tok.lower()
        if tok in string.punctuation:
            continue
        tok = tok.translate(str.maketrans('', '', string.punctuation))
        if tok == '':
            continue
        if tok.isdigit():
            continue
        if tok in stop_words:
            continue
        if len(tok) < 2:
            continue
        cleaned.append(tok)
    return cleaned

# Demo
filtered_tokens = filter_tokens(word_tokens)
print('Before Filtration:\n', word_tokens)
print('\nAfter Filtration:\n', filtered_tokens)

Before Filtration:
 ['Amazing', 'stay', '!', 'The', 'place', 'felt', 'very', 'cozy', 'for', '4', 'guests', '.', 'Check-in', 'was', 'smooth', 'and', 'the', 'amenities', 'were', 'exactly', 'what', 'we', 'needed', '.']

After Filtration:
 ['amazing', 'stay', 'place', 'felt', 'cozy', 'guests', 'checkin', 'smooth', 'amenities', 'exactly', 'needed']


In [6]:
# Regex allowing standard English letters, digits, whitespace and common punctuation
VALID_SCRIPT_PATTERN = re.compile(r"^[A-Za-z0-9\s.,!?'\"()\-:;/&%$#@*+=]*$")

def is_valid_script(text):
    """Return True if the text contains only valid English/Latin script characters."""
    if not isinstance(text, str) or text.strip() == '':
        return False
    return bool(VALID_SCRIPT_PATTERN.match(text))

def invalid_char_ratio(text):
    """Fraction of characters in text that fall outside the valid script pattern."""
    if not isinstance(text, str) or len(text) == 0:
        return 1.0
    invalid_chars = [ch for ch in text if not VALID_SCRIPT_PATTERN.match(ch)]
    return len(invalid_chars) / len(text)

# Demo
examples = [
    df['review_text'].iloc[0],
    'Ã¢ÅË 5minAeropuerto',
    "Bonjour, l'appartement etait tres bien!",
    'Great stay!! 5/5 :)'
]
for ex in examples:
    print(f'{ex!r:60} -> valid_script={is_valid_script(ex)}, invalid_ratio={invalid_char_ratio(ex):.2f}')

'Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed.' -> valid_script=True, invalid_ratio=0.00
'Ã¢ÅË 5minAeropuerto'                                        -> valid_script=False, invalid_ratio=0.21
"Bonjour, l'appartement etait tres bien!"                    -> valid_script=True, invalid_ratio=0.00
'Great stay!! 5/5 :)'                                        -> valid_script=True, invalid_ratio=0.00


In [7]:
def preprocess_review(text):
    """Full pipeline: tokenize -> filter -> validate script."""
    valid = is_valid_script(text)
    _, word_tokens = tokenize_text(text)
    filtered = filter_tokens(word_tokens)
    return pd.Series({
        'word_tokens': word_tokens,
        'filtered_tokens': filtered,
        'token_count': len(word_tokens),
        'filtered_token_count': len(filtered),
        'is_valid_script': valid,
        'invalid_char_ratio': invalid_char_ratio(text)
    })

processed = df['review_text'].apply(preprocess_review)
df_processed = pd.concat([df[['review_id', 'review_text']], processed], axis=1)
df_processed.head(10)

,review_id,review_text,word_tokens,filtered_tokens,token_count,filtered_token_count,is_valid_script,invalid_char_ratio
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[Amazing, stay, !, The, place, felt, very, coz...","[amazing, stay, place, felt, cozy, guests, che...",24,11,True,0.0
1,490116563,It was okay for the price. Location in XIII Au...,"[It, was, okay, for, the, price, ., Location, ...","[okay, price, location, xiii, aurelia, conveni...",15,7,True,0.0
2,582235668,Loved every minute of it. Our superhost was su...,"[Loved, every, minute, of, it, ., Our, superho...","[loved, every, minute, superhost, super, respo...",41,22,True,0.0
3,68054683,Decent stay overall. Some things could be impr...,"[Decent, stay, overall, ., Some, things, could...","[decent, stay, overall, things, could, improve...",17,10,True,0.0
4,248483824,Reasonable for a short trip. Location in Long ...,"[Reasonable, for, a, short, trip, ., Location,...","[reasonable, short, trip, location, long, isla...",15,9,True,0.0
5,155617131,Decent stay overall. It served its purpose for...,"[Decent, stay, overall, ., It, served, its, pu...","[decent, stay, overall, served, purpose, stay,...",14,7,True,0.0
6,710244614,"Nothing special, but fine. Our superhost was p...","[Nothing, special, ,, but, fine, ., Our, super...","[nothing, special, fine, superhost, polite, sl...",32,16,True,0.0
7,299174484,We had a rough experience. The location in Enc...,"[We, had, a, rough, experience, ., The, locati...","[rough, experience, location, enclosstlaurent,...",37,16,True,0.0
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[Perfect, for, our, trip, ., Check-in, was, sm...","[perfect, trip, checkin, smooth, amenities, ex...",39,18,True,0.0
9,469473761,Would not recommend. The private room in house...,"[Would, not, recommend, ., The, private, room,...","[would, recommend, private, room, house, clean...",30,13,True,0.0


In [9]:
print('Total reviews:', len(df_processed))

Total reviews: 15000


In [10]:
print('Reviews with valid script:', df_processed['is_valid_script'].sum())

Reviews with valid script: 14985


In [11]:
print('Reviews with invalid/garbled script:', (~df_processed['is_valid_script']).sum())

Reviews with invalid/garbled script: 15


In [14]:
print('\nAverage tokens per review (before filtration):', round(df_processed['token_count'].mean(), 2))
print('Average tokens per review (after filtration):', round(df_processed['filtered_token_count'].mean(), 2))


Average tokens per review (before filtration): 29.13
Average tokens per review (after filtration): 13.66


In [15]:
df_processed[~df_processed['is_valid_script']][['review_id', 'review_text']].head()

,review_id,review_text
2312,62076203,Decent stay overall. Location in Alvaro Obreg...
2401,122891328,Reasonable for a short trip. Location in Alva...
3746,243384892,Below our expectations. The location in Alvar...
6121,348415567,Not worth the price. The location in Alvaro O...
6446,274276424,We had a rough experience. Check-in was confus...


In [16]:
df_processed.to_csv('Preprocessed_Airbnb_Reviews.csv', index=False)
print('Saved to Preprocessed_Airbnb_Reviews.csv')

Saved to Preprocessed_Airbnb_Reviews.csv
